In [24]:
import random
from collections import defaultdict
import heapq
import matplotlib.pyplot as plt
import networkx as nx


In [25]:
# Load Graph from edges file
# -----------------------------
file_path = "C:/Users/Administrator/Desktop/ENZYMES_g100/ENZYMES_g100.edges"  # Adjust if needed
edges = []
nodes_set = set()

with open(file_path, "r") as f:
    for line in f:
        u, v = map(int, line.strip().split())
        edges.append((u, v, 1))  # assume default weight = 1
        nodes_set.update([u, v])

n = max(nodes_set) + 1  # total number of nodes

In [26]:
# Union Find (for Kruskal, Borůvka, Reverse-Delete)
# -----------------------------
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    
    def find(self, u):
        while self.parent[u] != u:
            self.parent[u] = self.parent[self.parent[u]]
            u = self.parent[u]
        return u

    def union(self, u, v):
        pu, pv = self.find(u), self.find(v)
        if pu != pv:
            self.parent[pu] = pv
            return True
        return False

In [27]:
# Kruskal's Algorithm
# -----------------------------
def kruskal(n, edges):
    edges.sort(key=lambda x: x[2])
    uf = UnionFind(n)
    mst = []
    total_weight = 0
    for u, v, w in edges:
        if uf.union(u, v):
            mst.append((u, v, w))
            total_weight += w
    return mst, total_weight

In [28]:
def prim(n, edges):
    adj = defaultdict(list)
    for u, v, w in edges:
        adj[u].append((w, v, u))
        adj[v].append((w, u, v))

    visited = [False] * n
    total_weight = 0
    mst = []

    for start in range(n):
        if not visited[start] and start in adj:
            min_heap = [(0, start, -1)]
            while min_heap:
                w, u, parent = heapq.heappop(min_heap)
                if visited[u]:
                    continue
                visited[u] = True
                total_weight += w
                if parent != -1:
                    mst.append((parent, u, w))
                for next_w, v, from_u in adj[u]:
                    if not visited[v]:
                        heapq.heappush(min_heap, (next_w, v, u))
    
    return mst, total_weight

In [29]:
# Borůvka's Algorithm (Optimized)
# -----------------------------
def boruvka_optimized(n, edges):
    uf = UnionFind(n)
    mst = []
    total_weight = 0
    num_components = n
    
    adj = defaultdict(list)
    for idx, (u, v, w) in enumerate(edges):
        adj[u].append((v, w, idx))
        adj[v].append((u, w, idx))
    
    while num_components > 1:
        cheapest = [(-1, float('inf'))] * n  # (edge_index, weight)
        changed = False
        
        for u in range(n):
            if uf.find(u) != u:  # Not the component's root
                continue
                
            for v, w, idx in adj[u]:
                root_v = uf.find(v)
                if root_v != u:  # Edge goes to different component
                    if w < cheapest[u][1]:
                        cheapest[u] = (idx, w)
        
        for u in range(n):
            if cheapest[u][0] != -1:
                idx = cheapest[u][0]
                u_edge, v_edge, w_edge = edges[idx]
                if uf.union(u_edge, v_edge):
                    mst.append((u_edge, v_edge, w_edge))
                    total_weight += w_edge
                    num_components -= 1
                    changed = True
        
        if not changed:
            break
    
    return mst, total_weight


In [42]:
# Reverse-Delete Algorithm
# -----------------------------
def reverse_delete(n, edges):
    edges_sorted = sorted(edges.copy(), key=lambda x: -x[2])
    mst_edges = edges.copy()

    def is_connected(nodes_in_graph, current_edges):
        uf = UnionFind(n)
        for u, v, _ in current_edges:
            uf.union(u, v)
        roots = set(uf.find(node) for node in nodes_in_graph)
        return len(roots) == 1

    # Only check connectivity among nodes that actually exist in the graph
    nodes_in_graph = set()
    for u, v, _ in edges:
        nodes_in_graph.add(u)
        nodes_in_graph.add(v)

    for u, v, w in edges_sorted:
        if (u, v, w) in mst_edges:
            mst_edges.remove((u, v, w))
            if not is_connected(nodes_in_graph, mst_edges):
                mst_edges.append((u, v, w))  # Re-add if graph becomes disconnected

    total_weight = sum(w for _, _, w in mst_edges)
    return mst_edges, total_weight

In [31]:
# Karger's Algorithm (Min Cut, not MST)
# -----------------------------
def karger_min_cut(n, edges, iterations=100):
    min_cut = float('inf')

    for _ in range(iterations):
        parent = list(range(n))
        e = edges[:]

        def find(u):
            while parent[u] != u:
                parent[u] = parent[parent[u]]
                u = parent[u]
            return u

        def union(u, v):
            pu, pv = find(u), find(v)
            if pu != pv:
                parent[pu] = pv

        vertices = n
        while vertices > 2:
            u, v, _ = random.choice(e)
            if find(u) != find(v):
                union(u, v)
                vertices -= 1
            e = [edge for edge in e if find(edge[0]) != find(edge[1])]

        cut = len([edge for edge in edges if find(edge[0]) != find(edge[1])])
        min_cut = min(min_cut, cut)
    
    return min_cut


In [43]:
# Run and print results
# -----------------------------
print("Running MST algorithms on ENZYMES_g100.edges\n")

kr_mst, kr_cost = kruskal(n, edges)
print(f"Kruskal's: total cost = {kr_cost}, edges = {len(kr_mst)}")

pr_mst, pr_cost = prim(n, edges)
print(f"Prim's: total cost = {pr_cost}, edges = {len(pr_mst)}")

bo_mst, bo_cost = boruvka_optimized(n, edges)
print(f"Borůvka's: total cost = {bo_cost}, edges = {len(bo_mst)}")

rd_mst, rd_cost = reverse_delete(n, edges)
print(f"Reverse-Delete: total cost = {rd_cost}, edges = {len(rd_mst)}")

min_cut = karger_min_cut(n, edges, iterations=100)
print(f"Karger's Min Cut estimate = {min_cut}")

Running MST algorithms on ENZYMES_g100.edges

Kruskal's: total cost = 4, edges = 4
Prim's: total cost = 4, edges = 4
Borůvka's: total cost = 4, edges = 4
Reverse-Delete: total cost = 4, edges = 4
Karger's Min Cut estimate = 0


In [62]:
import os
import random
import heapq
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from collections import defaultdict
import time

# --------------------------
# Union-Find with operation counting
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n
        self.operations = 0  # Track number of operations
    
    def find(self, u):
        self.operations += 1
        if self.parent[u] != u:
            self.parent[u] = self.find(self.parent[u])  # Path compression
        return self.parent[u]
    
    def union(self, u, v):
        self.operations += 1
        pu, pv = self.find(u), self.find(v)
        if pu != pv:
            # Union by rank
            if self.rank[pu] < self.rank[pv]:
                self.parent[pu] = pv
            elif self.rank[pu] > self.rank[pv]:
                self.parent[pv] = pu
            else:
                self.parent[pv] = pu
                self.rank[pu] += 1
            return True
        return False

# --------------------------
# Load dataset with proper node handling
def load_graph(file_path):
    edges = []
    nodes = set()
    with open(file_path, "r") as f:
        for line in f:
            line = line.strip()
            if line:
                u, v = map(int, line.split())
                edges.append((u, v, 1))  # assume weight = 1
                nodes.update([u, v])
    
    # Create mapping from actual nodes to 0-based indices
    node_list = sorted(list(nodes))
    node_to_idx = {node: i for i, node in enumerate(node_list)}
    idx_to_node = {i: node for i, node in enumerate(node_list)}
    
    # Convert edges to use 0-based indices
    indexed_edges = [(node_to_idx[u], node_to_idx[v], w) for u, v, w in edges]
    
    return indexed_edges, len(nodes), nodes, node_to_idx, idx_to_node

# --------------------------
# Kruskal Algorithm with step and cost tracking
def kruskal_steps(n, edges):
    start_time = time.time()
    sorted_edges = sorted(edges, key=lambda x: x[2])  # O(E log E)
    sorting_cost = len(edges) * (len(edges).bit_length() - 1)  # Approximation of E log E
    
    uf = UnionFind(n)
    steps = []
    mst_edges = []
    total_comparisons = 0
    
    for i, (u, v, w) in enumerate(sorted_edges):
        total_comparisons += 1
        uf_ops_before = uf.operations
        
        # Record the step of considering this edge
        step = {
            'algorithm': 'Kruskal',
            'step': i,
            'considering': (u, v),
            'current_mst': mst_edges.copy(),
            'action': None,
            'total_edges': len(sorted_edges),
            'computational_cost': {
                'sorting_cost': sorting_cost,
                'union_find_ops': uf.operations,
                'total_comparisons': total_comparisons,
                'time_complexity': f"O(E log E + E α(V))",
                'current_operation': f"Union-Find on edge ({u}, {v})"
            }
        }
        
        if uf.union(u, v):
            mst_edges.append((u, v))
            step['action'] = 'added'
            step['current_mst'] = mst_edges.copy()
        else:
            step['action'] = 'rejected (creates cycle)'
        
        step['computational_cost']['union_find_ops'] = uf.operations
        steps.append(step)
    
    execution_time = time.time() - start_time
    for step in steps:
        step['computational_cost']['execution_time'] = execution_time
    
    return steps

# --------------------------
# Prim Algorithm with step and cost tracking
def prim_steps(n, edges):
    start_time = time.time()
    adj = defaultdict(list)
    graph_construction_ops = 0
    
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
        graph_construction_ops += 2
    
    visited = [False] * n
    steps = []
    mst_edges = []
    heap_operations = 0
    
    # Start from node 0 (or first available node)
    start_node = 0
    if start_node not in adj:
        for node in range(n):
            if node in adj:
                start_node = node
                break
    
    heap = [(0, start_node, -1)]
    heap_operations += 1  # Initial push
    step_count = 0
    
    while heap:
        heap_operations += 1  # Pop operation
        weight, u, parent = heapq.heappop(heap)
        
        if visited[u]:
            continue
            
        visited[u] = True
        
        if parent != -1:
            mst_edges.append((parent, u))
            steps.append({
                'algorithm': 'Prim',
                'step': step_count,
                'considering': (parent, u),
                'current_mst': mst_edges.copy(),
                'action': 'added',
                'visited': visited.copy(),
                'computational_cost': {
                    'graph_construction': graph_construction_ops,
                    'heap_operations': heap_operations,
                    'vertices_processed': sum(visited),
                    'time_complexity': "O(E log V)",
                    'current_operation': f"Processing vertex {u}"
                }
            })
            step_count += 1
        
        # Add all edges from current node to heap
        for v, edge_weight in adj[u]:
            if not visited[v]:
                heapq.heappush(heap, (edge_weight, v, u))
                heap_operations += 1  # Push operation
    
    execution_time = time.time() - start_time
    for step in steps:
        step['computational_cost']['execution_time'] = execution_time
    
    return steps

# --------------------------
# Borůvka Algorithm with step and cost tracking
def boruvka_steps(n, edges):
    start_time = time.time()
    uf = UnionFind(n)
    steps = []
    mst_edges = []
    iteration = 0
    total_edge_examinations = 0
    
    while len(mst_edges) < n - 1:
        # Find cheapest edge for each component
        cheapest = {}
        edge_examinations_this_iteration = 0
        
        for u, v, w in edges:
            edge_examinations_this_iteration += 1
            total_edge_examinations += 1
            root_u, root_v = uf.find(u), uf.find(v)
            
            if root_u != root_v:
                if root_u not in cheapest or w < cheapest[root_u][1]:
                    cheapest[root_u] = ((u, v), w)
                if root_v not in cheapest or w < cheapest[root_v][1]:
                    cheapest[root_v] = ((u, v), w)
        
        # Add all cheapest edges
        edges_added_this_iteration = []
        union_operations_this_iteration = 0
        
        for component_root, (edge, weight) in cheapest.items():
            u, v = edge
            union_operations_this_iteration += 1
            if uf.union(u, v):
                mst_edges.append((u, v))
                edges_added_this_iteration.append((u, v))
        
        if edges_added_this_iteration:
            steps.append({
                'algorithm': 'Borůvka',
                'step': iteration,
                'iteration': iteration,
                'edges_added': edges_added_this_iteration,
                'current_mst': mst_edges.copy(),
                'action': f'Added {len(edges_added_this_iteration)} edges',
                'computational_cost': {
                    'iteration': iteration + 1,
                    'edge_examinations_this_iter': edge_examinations_this_iteration,
                    'total_edge_examinations': total_edge_examinations,
                    'union_find_ops': uf.operations,
                    'components_remaining': n - len(mst_edges),
                    'time_complexity': "O(E log V)",
                    'current_operation': f"Iteration {iteration + 1}: Found {len(cheapest)} cheapest edges"
                }
            })
            iteration += 1
        else:
            break
    
    execution_time = time.time() - start_time
    for step in steps:
        step['computational_cost']['execution_time'] = execution_time
    
    return steps

# --------------------------
# Reverse-Delete Algorithm with step and cost tracking
def reverse_delete_steps(n, edges):
    start_time = time.time()
    sorted_edges = sorted(edges, key=lambda x: -x[2])
    sorting_cost = len(edges) * (len(edges).bit_length() - 1)
    
    current_edges = edges.copy()
    steps = []
    connectivity_checks = 0
    
    def is_connected(edge_list, num_nodes):
        nonlocal connectivity_checks
        connectivity_checks += 1
        
        if not edge_list:
            return num_nodes <= 1
        
        uf = UnionFind(num_nodes)
        for u, v, _ in edge_list:
            uf.union(u, v)
        
        nodes_in_graph = set()
        for u, v, _ in edge_list:
            nodes_in_graph.add(u)
            nodes_in_graph.add(v)
        
        if not nodes_in_graph:
            return True
        
        first_node = next(iter(nodes_in_graph))
        root = uf.find(first_node)
        
        return all(uf.find(node) == root for node in nodes_in_graph)
    
    for i, (u, v, w) in enumerate(sorted_edges):
        if (u, v, w) in current_edges:
            current_edges.remove((u, v, w))
            
            connectivity_checks_before = connectivity_checks
            connected = is_connected(current_edges, n)
            
            if connected:
                steps.append({
                    'algorithm': 'Reverse-Delete',
                    'step': i,
                    'considering': (u, v),
                    'current_mst': [(x, y) for x, y, _ in current_edges],
                    'action': 'removed (redundant)',
                    'weight': w,
                    'computational_cost': {
                        'sorting_cost': sorting_cost,
                        'connectivity_checks': connectivity_checks,
                        'edges_remaining': len(current_edges),
                        'time_complexity': "O(E²)",
                        'current_operation': f"Connectivity check for edge ({u}, {v})"
                    }
                })
            else:
                current_edges.append((u, v, w))
                steps.append({
                    'algorithm': 'Reverse-Delete',
                    'step': i,
                    'considering': (u, v),
                    'current_mst': [(x, y) for x, y, _ in current_edges],
                    'action': 'kept (necessary)',
                    'weight': w,
                    'computational_cost': {
                        'sorting_cost': sorting_cost,
                        'connectivity_checks': connectivity_checks,
                        'edges_remaining': len(current_edges),
                        'time_complexity': "O(E²)",
                        'current_operation': f"Connectivity check for edge ({u}, {v})"
                    }
                })
    
    execution_time = time.time() - start_time
    for step in steps:
        step['computational_cost']['execution_time'] = execution_time
    
    return steps

# --------------------------
# Enhanced Visualization with Computational Cost
def visualize_algorithm(steps, nodes, idx_to_node, save_path, algorithm):
    # Create graph with original node labels
    G = nx.Graph()
    original_nodes = [idx_to_node[i] for i in range(len(nodes))]
    G.add_nodes_from(original_nodes)
    
    # Create layout
    pos = nx.spring_layout(G, seed=42, k=2, iterations=50)
    
    fig = plt.figure(figsize=(20, 12))
    
    # Create a 2x2 grid layout
    ax1 = plt.subplot2grid((2, 2), (0, 0))  # MST visualization
    ax2 = plt.subplot2grid((2, 2), (0, 1))  # Algorithm info
    ax3 = plt.subplot2grid((2, 2), (1, 0), colspan=2)  # Computational cost info
    
    def update(frame):
        ax1.clear()
        ax2.clear()
        ax3.clear()
        
        if frame >= len(steps):
            return
            
        step = steps[frame]
        
        # Top Left: Current MST
        G_mst = nx.Graph()
        G_mst.add_nodes_from(original_nodes)
        
        if 'current_mst' in step:
            mst_edges_original = [(idx_to_node[u], idx_to_node[v]) for u, v in step['current_mst']]
            G_mst.add_edges_from(mst_edges_original)
        
        # Draw MST
        nx.draw_networkx_nodes(G_mst, pos, ax=ax1, node_color='lightblue', 
                              node_size=500, alpha=0.8)
        nx.draw_networkx_edges(G_mst, pos, ax=ax1, edge_color='blue', 
                              width=2, alpha=0.7)
        nx.draw_networkx_labels(G_mst, pos, ax=ax1, font_size=12, font_weight='bold')
        
        # Highlight currently considering edge
        if 'considering' in step:
            u_idx, v_idx = step['considering']
            u_orig, v_orig = idx_to_node[u_idx], idx_to_node[v_idx]
            edge_color = 'green' if 'added' in step['action'] else 'red'
            nx.draw_networkx_edges(G_mst, pos, edgelist=[(u_orig, v_orig)], 
                                 edge_color=edge_color, width=4, alpha=0.8, ax=ax1)
        
        ax1.set_title(f"{algorithm} - Step {frame + 1}/{len(steps)}\nCurrent MST", 
                     fontsize=14, fontweight='bold')
        ax1.axis('off')
        
        # Top Right: Algorithm info
        y_pos = 0.9
        ax2.text(0.05, y_pos, f"Algorithm: {algorithm}", fontsize=14, fontweight='bold',
                transform=ax2.transAxes)
        y_pos -= 0.1
        ax2.text(0.05, y_pos, f"Step: {frame + 1}/{len(steps)}", fontsize=12,
                transform=ax2.transAxes)
        y_pos -= 0.1
        
        if 'considering' in step:
            u_idx, v_idx = step['considering']
            u_orig, v_orig = idx_to_node[u_idx], idx_to_node[v_idx]
            ax2.text(0.05, y_pos, f"Considering edge: ({u_orig}, {v_orig})", 
                    fontsize=12, transform=ax2.transAxes)
            y_pos -= 0.1
        
        if 'action' in step:
            action_color = 'green' if 'added' in step['action'] else 'red'
            ax2.text(0.05, y_pos, f"Action: {step['action']}", fontsize=12,
                    color=action_color, fontweight='bold', transform=ax2.transAxes)
            y_pos -= 0.1
        
        if 'current_mst' in step:
            ax2.text(0.05, y_pos, f"MST edges so far: {len(step['current_mst'])}", 
                    fontsize=12, transform=ax2.transAxes)
            y_pos -= 0.1
        
        # Algorithm specific info
        if algorithm == 'Borůvka' and 'iteration' in step:
            ax2.text(0.05, y_pos, f"Iteration: {step['iteration'] + 1}", 
                    fontsize=12, transform=ax2.transAxes)
        elif algorithm == 'Reverse-Delete' and 'weight' in step:
            ax2.text(0.05, y_pos, f"Edge weight: {step['weight']}", 
                    fontsize=12, transform=ax2.transAxes)
        
        ax2.set_xlim(0, 1)
        ax2.set_ylim(0, 1)
        ax2.axis('off')
        
        # Bottom: Computational Cost Analysis
        cost = step.get('computational_cost', {})
        
        # Title
        ax3.text(0.02, 0.9, "Computational Cost Analysis", fontsize=16, 
                fontweight='bold', transform=ax3.transAxes)
        
        # Time Complexity
        ax3.text(0.02, 0.8, f"Time Complexity: {cost.get('time_complexity', 'N/A')}", 
                fontsize=12, fontweight='bold', color='blue', transform=ax3.transAxes)
        
        # Current Operation
        ax3.text(0.02, 0.7, f"Current Operation: {cost.get('current_operation', 'N/A')}", 
                fontsize=11, transform=ax3.transAxes)
        
        # Algorithm-specific metrics
        y_start = 0.6
        if algorithm == 'Kruskal':
            metrics = [
                f"Sorting Cost: {cost.get('sorting_cost', 0)} operations",
                f"Union-Find Operations: {cost.get('union_find_ops', 0)}",
                f"Total Comparisons: {cost.get('total_comparisons', 0)}",
                f"Execution Time: {cost.get('execution_time', 0):.6f} seconds"
            ]
        elif algorithm == 'Prim':
            metrics = [
                f"Graph Construction: {cost.get('graph_construction', 0)} operations",
                f"Heap Operations: {cost.get('heap_operations', 0)}",
                f"Vertices Processed: {cost.get('vertices_processed', 0)}/{len(nodes)}",
                f"Execution Time: {cost.get('execution_time', 0):.6f} seconds"
            ]
        elif algorithm == 'Borůvka':
            metrics = [
                f"Current Iteration: {cost.get('iteration', 0)}",
                f"Edge Examinations (this iter): {cost.get('edge_examinations_this_iter', 0)}",
                f"Total Edge Examinations: {cost.get('total_edge_examinations', 0)}",
                f"Union-Find Operations: {cost.get('union_find_ops', 0)}",
                f"Components Remaining: {cost.get('components_remaining', 0)}",
                f"Execution Time: {cost.get('execution_time', 0):.6f} seconds"
            ]
        elif algorithm == 'Reverse-Delete':
            metrics = [
                f"Sorting Cost: {cost.get('sorting_cost', 0)} operations",
                f"Connectivity Checks: {cost.get('connectivity_checks', 0)}",
                f"Edges Remaining: {cost.get('edges_remaining', 0)}",
                f"Execution Time: {cost.get('execution_time', 0):.6f} seconds"
            ]
        else:
            metrics = []
        
        for i, metric in enumerate(metrics):
            ax3.text(0.02, y_start - i*0.08, f"• {metric}", fontsize=10, 
                    transform=ax3.transAxes)
        
        # Performance comparison box
        comparison_text = f"""
Performance Notes for {algorithm}:

Best Case: {get_best_case(algorithm)}
Average Case: {get_average_case(algorithm)}
Worst Case: {get_worst_case(algorithm)}

Space Complexity: {get_space_complexity(algorithm)}
        """
        
        ax3.text(0.55, 0.8, comparison_text, fontsize=10, 
                transform=ax3.transAxes, verticalalignment='top',
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))
        
        ax3.set_xlim(0, 1)
        ax3.set_ylim(0, 1)
        ax3.axis('off')
    
    # Create animation
    ani = animation.FuncAnimation(fig, update, frames=len(steps), 
                                interval=3000, repeat=True, blit=False)
    
    # Save animation
    writer = animation.PillowWriter(fps=0.33)
    ani.save(save_path, writer=writer)
    plt.close()
    print(f"Saved {algorithm} animation with computational cost → {save_path}")

def get_best_case(algorithm):
    cases = {
        'Kruskal': 'O(E log E)',
        'Prim': 'O(E log V)',
        'Borůvka': 'O(E log V)',
        'Reverse-Delete': 'O(E²)'
    }
    return cases.get(algorithm, 'N/A')

def get_average_case(algorithm):
    cases = {
        'Kruskal': 'O(E log E)',
        'Prim': 'O(E log V)',
        'Borůvka': 'O(E log V)',
        'Reverse-Delete': 'O(E²)'
    }
    return cases.get(algorithm, 'N/A')

def get_worst_case(algorithm):
    cases = {
        'Kruskal': 'O(E log E)',
        'Prim': 'O(V²) with array, O(E log V) with heap',
        'Borůvka': 'O(E log V)',
        'Reverse-Delete': 'O(E²)'
    }
    return cases.get(algorithm, 'N/A')

def get_space_complexity(algorithm):
    cases = {
        'Kruskal': 'O(V) for Union-Find',
        'Prim': 'O(V + E) for adjacency list + O(V) for heap',
        'Borůvka': 'O(V) for Union-Find',
        'Reverse-Delete': 'O(E) for edge storage'
    }
    return cases.get(algorithm, 'N/A')

# --------------------------
# Main execution
def main():
    file_path = "ENZYMES_g100.edges"
    
    if not os.path.exists(file_path):
        print(f"File not found: {file_path}")
        print("Please make sure the ENZYMES_g100.edges file is in the current directory.")
        return
    
    # Load graph data
    edges, n, nodes, node_to_idx, idx_to_node = load_graph(file_path)
    print(f"Loaded graph with {n} nodes and {len(edges)} edges")
    print(f"Nodes: {sorted(list(nodes))}")
    
    # Generate steps for each algorithm
    print("\nGenerating MST algorithm steps with computational cost tracking...")
    
    algorithms = [
        ("Kruskal", kruskal_steps(n, edges)),
        ("Prim", prim_steps(n, edges)),
        ("Borůvka", boruvka_steps(n, edges)),
        ("Reverse-Delete", reverse_delete_steps(n, edges))
    ]
    
    # Create visualizations
    print("\nGenerating MST algorithm visualizations with computational cost...")
    for name, steps in algorithms:
        if steps:
            output_file = f"{name.lower().replace('-', '_')}_mst_with_cost.gif"
            visualize_algorithm(steps, nodes, idx_to_node, output_file, name)
            print(f"  → {name}: {len(steps)} steps")
            
            # Print computational summary
            if steps and 'computational_cost' in steps[-1]:
                final_cost = steps[-1]['computational_cost']
                print(f"     Final execution time: {final_cost.get('execution_time', 0):.6f} seconds")
        else:
            print(f"  → {name}: No steps generated")
    
    print("\nAll visualizations with computational cost analysis completed!")
    
    # Print final MST and performance comparison
    print("\n" + "="*80)
    print("FINAL MST COMPARISON WITH COMPUTATIONAL COSTS")
    print("="*80)
    
    for name, steps in algorithms:
        if steps and 'current_mst' in steps[-1]:
            final_mst = steps[-1]['current_mst']
            final_edges_original = [(idx_to_node[u], idx_to_node[v]) for u, v in final_mst]
            final_cost = steps[-1].get('computational_cost', {})
            
            print(f"\n{name}:")
            print(f"  MST Edges: {len(final_mst)} - {sorted(final_edges_original)}")
            print(f"  Time Complexity: {final_cost.get('time_complexity', 'N/A')}")
            print(f"  Execution Time: {final_cost.get('execution_time', 0):.6f} seconds")
            
            # Algorithm-specific metrics
            if name == 'Kruskal':
                print(f"  Union-Find Operations: {final_cost.get('union_find_ops', 0)}")
                print(f"  Total Comparisons: {final_cost.get('total_comparisons', 0)}")
            elif name == 'Prim':
                print(f"  Heap Operations: {final_cost.get('heap_operations', 0)}")
                print(f"  Vertices Processed: {final_cost.get('vertices_processed', 0)}")
            elif name == 'Borůvka':
                print(f"  Total Iterations: {final_cost.get('iteration', 0)}")
                print(f"  Total Edge Examinations: {final_cost.get('total_edge_examinations', 0)}")
            elif name == 'Reverse-Delete':
                print(f"  Connectivity Checks: {final_cost.get('connectivity_checks', 0)}")

if __name__ == "__main__":
    main()

Loaded graph with 5 nodes and 18 edges
Nodes: [1, 2, 3, 4, 5]

Generating MST algorithm steps with computational cost tracking...

Generating MST algorithm visualizations with computational cost...
Saved Kruskal animation with computational cost → kruskal_mst_with_cost.gif
  → Kruskal: 18 steps
     Final execution time: 0.000000 seconds
Saved Prim animation with computational cost → prim_mst_with_cost.gif
  → Prim: 4 steps
     Final execution time: 0.000000 seconds
Saved Borůvka animation with computational cost → borůvka_mst_with_cost.gif
  → Borůvka: 1 steps
     Final execution time: 0.000000 seconds
Saved Reverse-Delete animation with computational cost → reverse_delete_mst_with_cost.gif
  → Reverse-Delete: 18 steps
     Final execution time: 0.001292 seconds

All visualizations with computational cost analysis completed!

FINAL MST COMPARISON WITH COMPUTATIONAL COSTS

Kruskal:
  MST Edges: 4 - [(2, 1), (3, 1), (4, 1), (5, 2)]
  Time Complexity: O(E log E + E α(V))
  Execution Ti